In [1]:
import pandas as pd
import numpy as np


SyntaxError: cannot assign to literal here. Maybe you meant '==' instead of '='? (931024692.py, line 3)

In [4]:
import os
import pathlib
from pathlib import Path

root_path = Path("/mnt/cluster/datasets/bowel_retraction/v1/")


In [6]:
episode_dirs = [d for d in root_path.iterdir() if d.is_dir() and (d.name != 'zarr')]
datasets_paths = [str(d / 'episode.csv') for d in episode_dirs]

In [25]:
import re
episode_df = pd.read_csv(datasets_paths[0])
episode_df['depth_img'] = episode_df['frameLeftRectifiedPath'].apply(
    lambda x: re.sub(r'(\d+)\.png$', r'depth_\1.npy', x.replace("framesLeftRectified", "depth"))
)
depth = np.load(episode_df['depth_img'][0])



KeyError: 'frameLeftPath'

In [23]:
depth.mean()

np.float32(1.2578006)

In [32]:
import os
import pathlib
from pathlib import Path

root_path = Path("/mnt/cluster/datasets/threading_il/v3/")
depth_path = Path("/mnt/cluster/workspaces/mazzalore/depth_results/")
# Get all episode directories excluding 'code'
episode_dirs = [d for d in root_path.iterdir() if d.is_dir() and (d.name != 'code' and d.name != '246')]
# Get paths to all dataset.csv files
datasets_paths = [str(d / 'smoothed_dataset.csv') for d in episode_dirs]
episode_nums = [int(Path(p).parent.name) for p in datasets_paths]
sorted_paths = [p for _, p in sorted(zip(episode_nums, datasets_paths))]
# Extract unique episode numbers from sorted_paths
episode_nums = sorted(set(int(Path(p).parent.name) for p in sorted_paths))

# Create corresponding directories in depth_path
for num in episode_nums:
    folder_path = depth_path / str(num) / "stereoDepth"
    folder_path.mkdir(parents=True, exist_ok=True)
    print(f"Created folder: {folder_path}")


Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/0/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/1/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/2/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/3/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/4/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/5/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/6/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/7/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/8/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/9/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/10/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_results/11/stereoDepth
Created folder: /mnt/cluster/workspaces/mazzalore/depth_result

In [17]:
!pip install joblib


In [29]:
import joblib
from PIL import Image
# Collect all depth image paths
depth_paths = []
input_paths = []
for path in sorted(sorted_paths, key=lambda x: int(Path(x).parent.name)):
    episode_df = pd.read_csv(path)
    episode_df['depth_img'] = episode_df['left_img'].apply(lambda x: x.replace("framesLeft", "stereoDepth").replace("/mnt/cluster/datasets/threading_il/v3/", "/mnt/cluster/workspaces/mazzalore/depth_results/"))
    input_paths.extend(episode_df['left_img'].apply(lambda x: x.replace("framesLeft", "stereoDepth")).tolist())
    depth_paths.extend(episode_df['depth_img'].tolist())




In [30]:
input_paths

['/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/0.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/1.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/2.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/3.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/4.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/5.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/6.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/7.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/8.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/9.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/10.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/11.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/12.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/13.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/14.png',
 '/mnt/cluster/datasets/threading_il/v3/0/stereoDepth/15.png',
 '

In [34]:
from tqdm import  tqdm
# Parallel processing with tqdm, ensuring alignment
with joblib.Parallel(n_jobs=-1) as parallel:
    parallel(
        joblib.delayed(lambda input_path, depth_path: Image.fromarray(np.load(input_path.replace(".png", ".npy")).astype(np.uint8), 'L').save(depth_path))(input_path, depth_path)
        for input_path, depth_path in tqdm(zip(input_paths, depth_paths), total=len(depth_paths), desc="Processing depth images")
    )






Processing depth images:   0%|          | 0/436870 [00:00<?, ?it/s]




Processing depth images:   0%|          | 32/436870 [00:00<2:28:29, 49.03it/s]




Processing depth images:   0%|          | 48/436870 [00:01<2:53:33, 41.95it/s]




Processing depth images:   0%|          | 64/436870 [00:07<19:16:12,  6.30it/s]




Processing depth images:   0%|          | 80/436870 [00:10<20:45:37,  5.84it/s]




Processing depth images:   0%|          | 96/436870 [00:13<21:18:44,  5.69it/s]




Processing depth images:   0%|          | 112/436870 [00:16<22:44:48,  5.33it/s]




Processing depth images:   0%|          | 128/436870 [00:20<23:50:02,  5.09it/s]




Processing depth images:   0%|          | 144/436870 [00:23<23:19:23,  5.20it/s]




Processing depth images:   0%|          | 160/436870 [00:25<22:14:13,  5.46it/s]




Processing depth images:   0%|          | 176/436870 [00:28<21:29:08,  5.65it/s]




Processing depth images:   0%|          | 192/436870 [00:31<21:56:36,  5.53it/s]

KeyboardInterrupt: 

In [12]:
dataset_path = "/mnt/cluster/datasets/threading_il/v3/1/dataset.csv"

dataset = pd.read_csv(dataset_path)
dataset['depth_img'] = dataset['left_img'].apply(lambda x: x.replace("framesLeft", "stereoDepth"))
dataset['left_img'] = dataset['left_img'].apply(lambda x: x.replace("Left", "LeftRectified"))
dataset['right_img'] = dataset['left_img'].apply(lambda x: x.replace("Right", "RightRectified"))


In [13]:
from time import time
from PIL import Image
t1 = time()
for left_path in dataset['left_img']:
    left_img = Image.open(left_path)
print(f"time loading PIL images for an episode is {time()-t1}")

t2 = time()

for depth_path in dataset['depth_img']:
    depth_array = np.load(depth_path.replace(".png", ".npy"))
    
print(f"time loading depth images for an episode is {time()-t2}")

time loading PIL images for an episode is 68.62844395637512
time loading depth images for an episode is 157.19822573661804


In [ ]:
dataset_stats_path = '/mnt/cluster/workspaces/mazzalore/iros/act_checkpoints/act_training_20250507/dataset_stats.pkl'

In [ ]:
import pickle
with open(dataset_stats_path, 'rb') as f:
    dataset_stats = pickle.load(f)
dataset_stats['depth_std']

In [ ]:
initial_pos = dataset.loc[0, ['abs_pos_x_t', 'abs_pos_y_t', 'abs_pos_z_t']].values
pos_cols = ['abs_pos_x_t', 'abs_pos_y_t', 'abs_pos_z_t']
dataset[pos_cols] = dataset[pos_cols].astype(float).values - initial_pos.astype(float)
print(dataset[['abs_pos_x_t', 'abs_pos_y_t', 'abs_pos_z_t']].isna().sum())
print(dataset[['abs_pos_x_t', 'abs_pos_y_t', 'abs_pos_z_t']].dtypes)


In [ ]:
dataset[["actionx", "actiony", "actionz"]] = np.nan_to_num(np.array(dataset[["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]].shift(-1)) -
                                                                   np.array(dataset[["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]]))
dataset[["pred_x", "pred_y", "pred_z"]] = dataset[
    ["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]
].shift(-1).ffill()
dataset[["pred_x", "pred_y", "pred_z"]]


In [ ]:
np.nan_to_num(dataset[["abs_pos_x_t", "abs_pos_y_t", "abs_pos_z_t"]].shift(-1))[-1]